<a href="https://colab.research.google.com/github/Mytros/DS_ML_homework/blob/main/HW_%D0%92%D0%B8%D0%BA%D0%BE%D1%80%D0%B8%D1%81%D1%82%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BF%D1%80%D0%BE%D0%BC%D0%BF%D1%82%D1%96%D0%B2_%D1%96_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%96%D0%B2_%D0%B2_Langchain_DKoval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [1]:
!pip -q install langchain langchain_openai huggingface_hub openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 9.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import json, os
from pathlib import Path

creds_path = Path("/content/drive/MyDrive/Colab Notebooks/creds.json")
creds = json.loads(creds_path.read_text())

# Export keys to environment
os.environ["OPENAI_API_KEY"] = creds.get("OPENAI_API_KEY", "")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = creds.get("HUGGINGFACEHUB_API_TOKEN", "")
os.environ["MISTRAL_API_KEY"] = creds.get("MISTRAL_API_KEY", "")
os.environ["SERPAPI_API_KEY"] = creds.get("SERPAPI_API_KEY", "")


In [ ]:
!pip install -q -U langchain-community langchain-openai


In [5]:
#  CHOOSE ONE PROVIDER
PROVIDER = "openai"        # "openai" | "huggingface"

#  OpenAI (ChatGPT-3.5/4)
def build_llm_openai(temperature=0.3, model="gpt-4o-mini"):
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model=model, temperature=temperature)

#  HuggingFace  Mistral
def build_llm_hf(temperature=0.3, repo_id="mistralai/Mistral-7B-Instruct-v0.2"):
    from langchain_community.llms import HuggingFaceHub
    return HuggingFaceHub(
        repo_id=repo_id,
        model_kwargs={
            "temperature": temperature,
            "max_new_tokens": 256
        },
    )

#  select LLM depending on provider
if PROVIDER == "openai":
    llm = build_llm_openai(temperature=0.3)
else:
    llm = build_llm_hf(temperature=0.3)

#  basic prompt
topic = "Quantum Computing"
prompt = f"""
Topic: {topic}.
Explain in simple words: 1) definition, 2) key advantages, 3) current research.
Keep it concise, max 200 characters. Output a single paragraph without lists.
"""

#  run inference with token usage tracking
if PROVIDER == "openai":
    from langchain.callbacks import get_openai_callback
    with get_openai_callback() as cb:
        response = llm.invoke(prompt)
        print(response if isinstance(response, str) else getattr(response, "content", response))
        print(f"\nToken usage: {cb.total_tokens} | Prompt tokens: {cb.prompt_tokens} | Completion tokens: {cb.completion_tokens}")
        print(f"Estimated cost: ${cb.total_cost:.6f}")
else:
    # HuggingFaceHub does not provide token usage
    response = llm.invoke(prompt)
    print(response if isinstance(response, str) else getattr(response, "content", response))



Quantum computing uses the principles of quantum mechanics to process information in ways traditional computers can't. Its key advantages include solving complex problems faster and enhancing security. Current research focuses on improving qubit stability and developing practical applications in fields like cryptography and drug discovery.

Token usage: 101 | Prompt tokens: 51 | Completion tokens: 50
Estimated cost: $0.000038


Answer:

Why temperature 0.3? More imagination than in lection's value of 0.1

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [6]:
#  Task 2: Parameterized prompt with LangChain
# Choose provider: "openai" or "huggingface"
PROVIDER = "openai"

#  Imports
from typing import List
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM builders
def build_llm_openai(temperature=0.3, model="gpt-4o-mini"):
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model=model, temperature=temperature)

def build_llm_hf(temperature=0.3, repo_id="mistralai/Mistral-7B-Instruct-v0.2"):
    from langchain_community.llms import HuggingFaceHub
    return HuggingFaceHub(
        repo_id=repo_id,
        model_kwargs={"temperature": temperature, "max_new_tokens": 256},
    )

# Select LLM
llm = build_llm_openai(temperature=0.3) if PROVIDER == "openai" else build_llm_hf(temperature=0.3)

#  Parameterized prompt template
prompt = PromptTemplate.from_template(
    "Explain in simple words the topic: '{topic}'. "
    "Cover: 1) definition, 2) key advantages or main ideas, 3) current applications or research. "
    "Keep it concise in 3–4 sentences."
)

# Build chain: Prompt -> LLM -> Parser
chain = prompt | llm | StrOutputParser()

# Topics to run
topics: List[str] = [
    "Bayesian methods in machine learning",
    "Transformers in machine learning",
    "Explainable AI"
]

#  Run chain for each topic
if PROVIDER == "openai":
    from langchain.callbacks import get_openai_callback
    with get_openai_callback() as cb:
        for t in topics:
            print(f"\n=== Topic: {t} ===")
            print(chain.invoke({"topic": t}))
        print("\n--- OpenAI usage summary ---")
        print(f"Prompt tokens: {cb.prompt_tokens} | Completion tokens: {cb.completion_tokens} | Total: {cb.total_tokens}")
        print(f"Estimated cost: ${cb.total_cost:.6f}")
else:
    for t in topics:
        print(f"\n=== Topic: {t} ===")
        print(chain.invoke({"topic": t}))



=== Topic: Bayesian methods in machine learning ===
Bayesian methods in machine learning are techniques that use Bayes' theorem to update the probability of a hypothesis as more evidence or information becomes available. One key advantage is that they provide a way to incorporate prior knowledge into the model, allowing for more informed predictions and uncertainty quantification. These methods are widely used in areas like natural language processing, image recognition, and medical diagnosis, where they help improve decision-making and adapt to new data. Current research often focuses on making these methods more efficient and scalable for large datasets.

=== Topic: Transformers in machine learning ===
Transformers in machine learning are a type of model designed to process and understand sequences of data, like words in a sentence, by focusing on the relationships between them. Their key advantages include the ability to handle long-range dependencies and parallel processing, makin



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [ ]:
#!pip install -q langchain_community duckduckgo_search
#!pip install -q duckduckgo-search
#!pip install -q -U langchain-huggingface ddgs duckduckgo-search
!pip install -q -U langchain langchain-huggingface duckduckgo-search ddgs huggingface_hub

In [8]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

"Barack Obama , the 44th president of the United States, was born on August 4, 1961, in Honolulu, Hawaii, [1] to Barack Obama , Sr. (1936–1982; born in Oriang' Kogelo of Rachuonyo North District, [2] Kenya) and Stanley Ann Dunham, known as Ann (1942–1995; born in Wichita, Kansas, United States). Mar 22, 2008 · His father was also Barack, and also Barry: he chose the nickname when he came to America from Kenya on a scholarship in 1959. His was a typical immigrant transition. What is the full name of President Barack Obama? The full name of President Barack Obama is Barack Hussein Obama II . His real full name is Barack Hussein Obama II. He was named after his biological father, who never had much contact with him during his life but left him with a very unusual name. How did Barack Obama get his name? Barack Obama is named after his father , who was a Kenyan economist (called under the same name). He’s first real given name is “Barak”, also spelled Baraq (Not to be confused with Barack 

In [9]:
# Task 3: Using agent for process automation
# Creating agent for searching scientific publications

#!pip install langchain langchain-community duckduckgo-search

from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.llms import Ollama
import re

class ScientificPublicationAgent:
    def __init__(self):
        """Initialize agent for searching scientific publications"""

        # Initialize local LLM (using Ollama)
        self.llm = Ollama(model="llama2", temperature=0.1)

        # Create search tool
        self.search_tool = self._create_search_tool()

        # Create agent
        self.agent = self._create_agent()

    def _create_search_tool(self):
        """Create search tool using DuckDuckGo"""

        def search_scientific_papers(query: str) -> str:
            """Function to search for scientific publications"""

            # Create DuckDuckGo searcher
            search = DuckDuckGoSearchRun()

            # Form query for scientific articles
            scientific_query = f"{query} site:arxiv.org OR site:scholar.google.com OR site:pubmed.ncbi.nlm.nih.gov OR \"research paper\" OR \"journal article\""

            try:
                results = search.run(scientific_query)
                return results
            except Exception as e:
                return f"Search error: {str(e)}"

        return Tool(
            name="Scientific_Paper_Search",
            description="Searches for scientific publications and research on the internet. Use to find articles and papers on specific topics.",
            func=search_scientific_papers
        )

    def _create_agent(self):
        """Create ReAct agent"""

        tools = [self.search_tool]

        # Create agent prompt
        prompt_template = """
        You are a specialized agent for searching scientific publications. Your goal is to find and analyze scientific articles on given topics.

        When given a task to find publications:
        1. Use available tools to search
        2. Analyze search results
        3. Extract key information about each publication
        4. Format structured list with titles, authors and brief descriptions

        You have access to the following tools:
        {tools}

        Use the following format:
        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]
        Action Input: the input to the action
        Observation: the result of the action
        ... (this Thought/Action/Action Input/Observation can repeat N times)
        Thought: I now know the final answer
        Final Answer: the final answer to the original input question

        Question: {input}
        Thought: {agent_scratchpad}
        """

        prompt = PromptTemplate(
            template=prompt_template,
            input_variables=["input", "agent_scratchpad"],
            partial_variables={
                "tools": "\n".join([f"{tool.name}: {tool.description}" for tool in tools]),
                "tool_names": ", ".join([tool.name for tool in tools])
            }
        )

        # Create agent
        agent = create_react_agent(
            llm=self.llm,
            tools=tools,
            prompt=prompt
        )

        # Create executor
        agent_executor = AgentExecutor(
            agent=agent,
            tools=tools,
            verbose=True,
            handle_parsing_errors=True,
            max_iterations=3
        )

        return agent_executor

    def search_publications(self, topic: str, count: int = 5) -> str:
        """Search publications on given topic"""

        query = f"""
        Find {count} latest scientific publications on topic "{topic}".

        For each publication provide:
        1. Article title
        2. Authors
        3. Brief description (2-3 sentences)
        4. Publication source (if available)
        5. Publication year (if available)

        Format results in structured way.
        """

        try:
            result = self.agent.invoke({"input": query})
            return result["output"]
        except Exception as e:
            return f"Agent error: {str(e)}"

# Simple alternative without LLM
class SimplePublicationSearcher:
    """Simple version for basic publication search"""

    def __init__(self):
        self.search = DuckDuckGoSearchRun()

    def search_publications(self, topic: str, count: int = 5):
        """Simple publication search"""

        print(f" Searching for {count} publications on topic: {topic}")

        # Form queries for different sources
        queries = [
            f"{topic} site:arxiv.org",
            f"{topic} site:scholar.google.com",
            f"{topic} research paper recent",
            f"{topic} scientific article 2023 2024"
        ]

        all_results = []

        for i, query in enumerate(queries):
            try:
                print(f"Executing query {i+1}/4: {query[:50]}...")
                results = self.search.run(query)
                all_results.append(f"=== Query {i+1} Results ===\n{results}\n")
            except Exception as e:
                print(f" Error in query {i+1}: {e}")

        return "\n".join(all_results)

    def format_results(self, raw_results: str) -> str:
        """Format search results for better readability"""

        # Basic formatting
        formatted = raw_results.replace("...", "\n...")
        formatted = re.sub(r'(https?://[^\s]+)', r'\n🔗 \1', formatted)

        return formatted

# Demo function
def main():
    """Main demonstration function"""

    print("Initializing Scientific Publication Agent...")
    print("="*60)

    # Create simple searcher
    searcher = SimplePublicationSearcher()

    print("Agent created successfully!")

    # Search for AI publications
    print("\n SEARCHING AI PUBLICATIONS")
    print("-" * 40)

    topic = "artificial intelligence"
    results = searcher.search_publications(topic, count=5)

    print(f"\n Search results for '{topic}':")
    print("-" * 50)
    formatted_results = searcher.format_results(results)
    print(formatted_results)

    # Search for machine learning publications
    print("\n SEARCHING MACHINE LEARNING PUBLICATIONS")
    print("-" * 40)

    topic2 = "machine learning neural networks"
    results2 = searcher.search_publications(topic2, count=3)

    print(f"\n Search results for '{topic2}':")
    print("-" * 50)
    formatted_results2 = searcher.format_results(results2)
    print(formatted_results2)

# Additional utility functions
def save_results_to_file(results: str, filename: str = "research_results.txt"):
    """Save results to file"""

    with open(filename, 'w', encoding='utf-8') as f:
        f.write("SCIENTIFIC PUBLICATIONS SEARCH RESULTS\n")
        f.write("="*50 + "\n\n")
        f.write(results)

    print(f"Results saved to file: {filename}")

# Run the demo
if __name__ == "__main__":
    main()


Initializing Scientific Publication Agent...
Agent created successfully!

 SEARCHING AI PUBLICATIONS
----------------------------------------
 Searching for 5 publications on topic: artificial intelligence
Executing query 1/4: artificial intelligence site:arxiv.org...
Executing query 2/4: artificial intelligence site:scholar.google.com...
Executing query 3/4: artificial intelligence research paper recent...
Executing query 4/4: artificial intelligence scientific article 2023 20...

 Search results for 'artificial intelligence':
--------------------------------------------------
=== Query 1 Results ===
1 week ago - The encounter of artificial intelligence with consciousness research is often framed as a challenge: could this science determine whether such systems are conscious? We suggest it is equally an opportunity to expand and test the scope of existing theories of consciousness. 2 weeks ago - This community paper developed out of the NSF Workshop on the Future of Artificial Intelli



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [10]:
# Task 4: Creating business analytics assistant agent
# Simple agent for sales forecasting with weather and economic data

#!pip install langchain langchain-community duckduckgo-search pandas matplotlib numpy

from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.llms import FakeListLLM
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

class BusinessAnalyticsAgent:
    def __init__(self):
        """Initialize simple business analytics agent"""

        # Simple LLM for basic reasoning (no API needed)
        self.llm = FakeListLLM(responses=[
            "I need to search for current weather conditions in Brazil and global economic situation for oranges.",
            "Based on the data, I can analyze trends and make a forecast.",
            "Let me calculate the forecast using the historical data and external factors."
        ])

        # Create tools
        self.search_tool = self._create_search_tool()
        self.analytics_tool = self._create_analytics_tool()

        # Create agent
        self.agent = self._create_simple_agent()

    def _create_search_tool(self):
        """Create search tool for market data"""

        def search_market_data(query: str) -> str:
            """Search for market and weather data"""

            search = DuckDuckGoSearchRun()

            try:
                results = search.run(query)
                return results[:1000]  # Limit results length
            except Exception as e:
                return f"Search error: {str(e)}"

        return Tool(
            name="Market_Data_Search",
            description="Search for market data, weather conditions, economic indicators and industry trends.",
            func=search_market_data
        )

    def _create_analytics_tool(self):
        """Create analytics calculation tool"""

        def calculate_forecast(data_text: str) -> str:
            """Calculate sales forecast based on historical data"""

            try:
                # Extract numbers from text using regex
                numbers = re.findall(r'(\d{4})[^\d]*(\d+)', data_text)

                if len(numbers) >= 2:
                    years = [int(n[0]) for n in numbers]
                    sales = [int(n[1]) for n in numbers]

                    # Create DataFrame
                    df = pd.DataFrame({'Year': years, 'Sales': sales})

                    # Simple trend calculation
                    avg_growth = np.mean(np.diff(sales))
                    last_year_sales = sales[-1]
                    forecast_2025 = last_year_sales + avg_growth

                    # Create visualization
                    plt.figure(figsize=(10, 6))
                    plt.plot(years, sales, 'bo-', label='Historical Sales')
                    plt.plot([2024, 2025], [last_year_sales, forecast_2025], 'ro--', label='Forecast')
                    plt.title('Orange Export Forecast')
                    plt.xlabel('Year')
                    plt.ylabel('Sales (tons)')
                    plt.legend()
                    plt.grid(True)
                    plt.show()

                    result = f"""
BUSINESS ANALYTICS REPORT
========================

Historical Data Analysis:
- Years: {years}
- Sales: {sales} tons
- Average annual growth: {avg_growth:.1f} tons

Forecast for 2025:
- Projected sales: {forecast_2025:.1f} tons
- Growth trend: {'Positive' if avg_growth > 0 else 'Negative'}

Confidence Level: Medium (based on historical trend only)
                    """

                    return result
                else:
                    return "Could not extract sufficient historical data for analysis."

            except Exception as e:
                return f"Analytics error: {str(e)}"

        return Tool(
            name="Sales_Analytics",
            description="Perform sales forecasting and trend analysis on historical data.",
            func=calculate_forecast
        )

    def _create_simple_agent(self):
        """Create simple agent without complex LLM"""

        tools = [self.search_tool, self.analytics_tool]

        # Simple agent that just uses tools in sequence
        class SimpleAgent:
            def __init__(self, tools):
                self.tools = {tool.name: tool for tool in tools}
                self.search_tool = self.tools["Market_Data_Search"]
                self.analytics_tool = self.tools["Sales_Analytics"]

            def invoke(self, query_data):
                user_query = query_data["input"]

                print(" STEP 1: Searching for market data...")

                # Search for Brazil weather data
                weather_query = "Brazil weather conditions oranges 2024 2025 forecast"
                weather_data = self.search_tool.func(weather_query)

                # Search for global orange market
                market_query = "global orange market demand prices 2024 2025"
                market_data = self.search_tool.func(market_query)

                print(" STEP 2: Analyzing historical data...")

                # Analyze historical data
                forecast_result = self.analytics_tool.func(user_query)

                print(" STEP 3: Generating final report...")

                final_report = f"""
COMPREHENSIVE BUSINESS FORECAST REPORT
=====================================

{forecast_result}

EXTERNAL FACTORS ANALYSIS:
--------------------------

Weather Conditions in Brazil:
{weather_data[:300]}...

Global Market Situation:
{market_data[:300]}...

RECOMMENDATIONS:
--------------
1. Monitor weather conditions closely as they directly affect orange production
2. Track global demand trends and economic indicators
3. Consider diversifying markets if needed
4. Plan for potential supply chain adjustments

RISK FACTORS:
-------------
- Weather dependency (droughts, floods)
- Global economic fluctuations
- Competition from other citrus producers
- Currency exchange rate changes
                """

                return {"output": final_report}

        return SimpleAgent(tools)

    def analyze_business_case(self, user_input: str) -> str:
        """Main function to analyze business case"""

        try:
            result = self.agent.invoke({"input": user_input})
            return result["output"]
        except Exception as e:
            return f"Analysis failed: {str(e)}\nPlease check your data format and try again."

# Simple version without complex agent
class SimpleBusinessAnalyzer:
    """Simplified business analyzer"""

    def __init__(self):
        self.search = DuckDuckGoSearchRun()

    def analyze_case(self, user_input: str):
        """Analyze business case with simple approach"""

        print(" BUSINESS ANALYTICS AGENT STARTING...")
        print("="*50)

        # Step 1: Extract historical data
        print(" Step 1: Extracting historical data...")
        historical_data = self._extract_data(user_input)

        # Step 2: Search for external factors
        print(" Step 2: Searching for market conditions...")
        weather_info = self._search_weather_data()
        market_info = self._search_market_data()

        # Step 3: Generate forecast
        print(" Step 3: Generating forecast...")
        forecast = self._calculate_forecast(historical_data)

        # Step 4: Create final report
        report = self._generate_report(historical_data, forecast, weather_info, market_info)

        return report

    def _extract_data(self, text):
        """Extract numerical data from text"""

        # Find year-value pairs
        patterns = [
            r'в (\d{4})[^0-9]*(\d+)т',  # Ukrainian pattern
            r'(\d{4})[^0-9]*(\d+)т',    # Alternative pattern
            r'(\d{4})[^0-9]*(\d+)\s*tons?'  # English pattern
        ]

        data = {}
        for pattern in patterns:
            matches = re.findall(pattern, text)
            for year, amount in matches:
                data[int(year)] = int(amount)

        return data

    def _search_weather_data(self):
        """Search for Brazil weather conditions"""

        try:
            query = "Brazil orange weather forecast 2025 climate conditions"
            results = self.search.run(query)
            return results[:400]
        except:
            return "Weather data unavailable"

    def _search_market_data(self):
        """Search for global orange market data"""

        try:
            query = "global orange market demand prices 2025 forecast"
            results = self.search.run(query)
            return results[:400]
        except:
            return "Market data unavailable"

    def _calculate_forecast(self, data):
        """Calculate simple forecast"""

        if len(data) < 2:
            return "Insufficient data for forecast"

        years = sorted(data.keys())
        sales = [data[year] for year in years]

        # Calculate trend
        if len(sales) >= 3:
            # Use last 3 years for trend
            recent_sales = sales[-3:]
            recent_years = years[-3:]

            # Linear trend calculation
            x = np.array(range(len(recent_sales)))
            y = np.array(recent_sales)

            # Simple linear regression
            slope = np.polyfit(x, y, 1)[0]

            # Forecast for 2025
            forecast_2025 = sales[-1] + slope

            return {
                'forecast_2025': max(0, int(forecast_2025)),
                'trend': 'Increasing' if slope > 0 else 'Decreasing',
                'slope': slope,
                'historical': data
            }
        else:
            # Simple average growth
            avg_growth = np.mean(np.diff(sales))
            forecast_2025 = sales[-1] + avg_growth

            return {
                'forecast_2025': max(0, int(forecast_2025)),
                'trend': 'Stable',
                'slope': avg_growth,
                'historical': data
            }

    def _generate_report(self, historical_data, forecast, weather_info, market_info):
        """Generate comprehensive business report"""

        if isinstance(forecast, str):
            return f"ANALYSIS FAILED: {forecast}"

        report = f"""
 ORANGE EXPORT BUSINESS ANALYSIS REPORT
==========================================

 HISTORICAL DATA:
------------------
"""

        for year, amount in sorted(historical_data.items()):
            report += f"• {year}: {amount} tons\n"

        report += f"""
 FORECAST FOR 2025:
--------------------
• Predicted export volume: {forecast['forecast_2025']} tons
• Trend: {forecast['trend']}
• Growth rate: {forecast['slope']:.1f} tons/year

 WEATHER CONDITIONS IN BRAZIL:
--------------------------------
{weather_info}

 GLOBAL MARKET SITUATION:
---------------------------
{market_info}

 BUSINESS RECOMMENDATIONS:
----------------------------
1. Based on historical trend, expect {"growth" if forecast['slope'] > 0 else "decline"} in exports
2. Monitor Brazilian weather patterns closely
3. Stay informed about global orange demand
4. Consider market diversification strategies
5. Plan logistics for approximately {forecast['forecast_2025']} tons

  RISK FACTORS TO CONSIDER:
----------------------------
• Weather-dependent production in Brazil
• Global economic conditions affecting demand
• Currency exchange rate fluctuations
• Competition from other citrus exporters

 CONFIDENCE LEVEL: Medium
(Based on historical data and current market research)
        """

        return report

# Demo function
def main():
    print("="*60)

    # Create analyzer
    analyzer = SimpleBusinessAnalyzer()

    # Test case - orange export from Brazil
    user_query = """
    Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т,
    в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми
    зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії
    і попит на апельсини в світі виходячи з економічної ситуації.
    """

    print(" USER QUERY:")
    print("-" * 40)
    print(user_query)
    print("\n" + "="*60)

    # Analyze the case
    result = analyzer.analyze_case(user_query)

    print(" ANALYSIS RESULT:")
    print("-" * 40)
    print(result)

# Additional utility function
def test_different_cases():
    """Test agent with different business cases"""

    analyzer = SimpleBusinessAnalyzer()

    test_cases = [
        """
        We export coffee from Colombia. In 2021 we exported 150 tons,
        in 2022 - 180 tons, in 2023 - 165 tons, in 2024 - 200 tons.
        What should we expect for 2025?
        """,
        """
        Our company sells winter jackets. 2021: 1000 units, 2022: 1200 units,
        2023: 800 units, 2024: 1100 units. Forecast for 2025?
        """
    ]

    for i, case in enumerate(test_cases, 1):
        print(f"\n TEST CASE {i}:")
        print("="*50)
        result = analyzer.analyze_case(case)
        print(result)
        print("\n" + "="*50)

# Run the demo
if __name__ == "__main__":
    main()

print("Business Analytics Agent is ready!")
print("This agent can analyze sales data and provide forecasts with market research")

 USER QUERY:
----------------------------------------

    Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т,
    в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми
    зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії
    і попит на апельсини в світі виходячи з економічної ситуації.
    

 BUSINESS ANALYTICS AGENT STARTING...
 Step 1: Extracting historical data...
 Step 2: Searching for market conditions...
 Step 3: Generating forecast...
 ANALYSIS RESULT:
----------------------------------------

 ORANGE EXPORT BUSINESS ANALYSIS REPORT

 HISTORICAL DATA:
------------------
• 2021: 200 tons
• 2022: 190 tons
• 2023: 210 tons
• 2024: 220 tons

 FORECAST FOR 2025:
--------------------
• Predicted export volume: 235 tons
• Trend: Increasing
• Growth rate: 15.0 tons/year

 WEATHER CONDITIONS IN BRAZIL:
--------------------------------
Owing to better climate conditions , most orange trees in the citrus planted are

**What can be improved:**

1. **Forecasting accuracy** - use more complex statistical models instead of simple linear trend

2. **External factors processing** - better integrate found weather and market information into forecast calculations

3. **Data validation** - add quality and relevance checking for found information

4. **Structured search** - use specialized APIs for economic and weather data instead of general search

5. **Error handling** - improve stability when search services are unavailable

The agent demonstrates basic functionality for business analytics but requires additional work for use in real projects.